In [0]:
import os
gett = os.getcwd()
data = os.path.join(gett, 'yellow_tripdata_2024-02.parquet')
print(data)
broze_data = spark.read.parquet(data)
broze_data.createOrReplaceTempView("yellow_taxi")
result_df = spark.sql("""
    SELECT 
    HOUR(tpep_pickup_datetime) AS pickup_hour,
    count(*) as counting,
    AVG(fare_amount) as avg_fare_amount
    FROM yellow_taxi
    WHERE passenger_count > 0 
      AND trip_distance > 0 
      AND tpep_pickup_datetime >= '2024-02-01' 
      AND tpep_pickup_datetime < '2024-03-01'
    GROUP BY pickup_hour
    order by pickup_hour
""")
result_df.createOrReplaceTempView("gold_peak_hours")
display(result_df)
# Write as Delta table instead of Parquet
delta_path = os.path.join(gett, 'gold_peak_hours')
result_df.write.mode("overwrite").format("delta").save(delta_path)
print(f"✅ Delta table saved to: {delta_path}")


/Workspace/Users/alwibuchori111@gmail.com/Drafts/yellow_tripdata_2024-02.parquet


pickup_hour,counting,avg_fare_amount
0,70677,17.660702208639442
1,47871,15.292689101961448
2,32499,14.732843779808572
3,21077,15.61222896996729
4,13354,22.355962258499197
5,14850,26.71203569023567
6,35378,21.20754367120804
7,75262,17.773035529218248
8,106481,16.91528479259234
9,119818,17.192525830843717


✅ Delta table saved to: /Workspace/Users/alwibuchori111@gmail.com/Drafts/gold_peak_hours


In [0]:
import os 
check = os.getcwd()
delta_path = f"file:{os.path.join(check, 'gold_peak_hours')}"

if os.path.exists(os.path.join(check, 'gold_peak_hours')):
  print("✅ Delta table exists!\n")
  
  # OPTIMIZE: Compact files and Z-order by pickup_hour
  print("Running OPTIMIZE with Z-ORDER...")
  optimize_result = spark.sql(f"OPTIMIZE delta.`{delta_path}` ZORDER BY (pickup_hour)")
  display(optimize_result)
  
  # VACUUM: Remove old files (retain 168 hours = 7 days)
  print("\nRunning VACUUM to clean up old files...")
  vacuum_result = spark.sql(f"VACUUM delta.`{delta_path}` RETAIN 168 HOURS")
  display(vacuum_result)
  
  print("\n🎉 Optimization complete!")
else:
  print("❌ Delta table does not exist. Run Cell 1 first.")



✅ Delta table exists!

Running OPTIMIZE with Z-ORDER...


path,metrics
file:/Workspace/Users/alwibuchori111@gmail.com/Drafts/gold_peak_hours,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1707), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1787027796872, 1787027797397, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"



Running VACUUM to clean up old files...


path
file:/Workspace/Users/alwibuchori111@gmail.com/Drafts/gold_peak_hours



🎉 Optimization complete!
